# Customer Segmentation & Market Basket Analysis (K-Means & Apriori)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BurakGuldal10/Machine-Learning-and-Data-Analysis-Projects/blob/main/Customer_Segmentation_and_Market_Basket_Analysis.ipynb)

## 📌 Proje Özeti / Project Overview
Bu projede, perakende sektöründe iki temel veri madenciliği yaklaşımı birleştirilmiştir:
1. **Müşteri Segmentasyonu (K-Means):** Alışveriş sıklığı, ortalama sepet tutarı ve ürün çeşitliliği kriterlerine göre müşterileri davranışsal kümelere ayırmak.
2. **Birliktelik Analizi (Market Basket Analysis - Apriori):** Sepette birlikte satın alınan ürünleri keşfederek **Support (Destek)**, **Confidence (Güven)** ve **Lift (Kaldıraç)** metrikleriyle çapraz satış kuralları üretmek.

### 1. Kütüphanelerin Yüklenmesi / Import Libraries

In [ ]:
# Gerekli kütüphaneleri yükleyelim
!pip -q install mlxtend

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

sns.set_theme(style="whitegrid")
print("Kütüphaneler hazır!")

### 2. Alışveriş Verisinin Oluşturulması / Dataset Generation

In [ ]:
np.random.seed(42)
n_musteri = 120
musteri_id = np.arange(1, n_musteri + 1)

# 3 farklı müşteri profili oluşturalım
# Grup 1: Seyrek gelen, düşük harcayan
# Grup 2: Düzenli gelen, orta harcayan
# Grup 3: VIP / Yüksek sıklık ve yüksek sepet tutarı
grup1 = pd.DataFrame({
    'Musteri_ID': musteri_id[:40],
    'Alisveris_Sikligi': np.random.poisson(lam=3, size=40),
    'Ortalama_Sepet_Tutari': np.random.normal(loc=150, scale=30, size=40),
    'Urun_Cesidi': np.random.randint(2, 5, size=40)
})

grup2 = pd.DataFrame({
    'Musteri_ID': musteri_id[40:80],
    'Alisveris_Sikligi': np.random.poisson(lam=8, size=40),
    'Ortalama_Sepet_Tutari': np.random.normal(loc=400, scale=50, size=40),
    'Urun_Cesidi': np.random.randint(4, 7, size=40)
})

grup3 = pd.DataFrame({
    'Musteri_ID': musteri_id[80:],
    'Alisveris_Sikligi': np.random.poisson(lam=15, size=40),
    'Ortalama_Sepet_Tutari': np.random.normal(loc=850, scale=100, size=40),
    'Urun_Cesidi': np.random.randint(6, 9, size=40)
})

musteriler = pd.concat([grup1, grup2, grup3], ignore_index=True)
print("Müşteri Davranış Verisi İlk 5 Satır:")
display(musteriler.head())

### 3. Müşteri Verisinin Görsel İncelemesi / Pairplot Inspection

In [ ]:
sns.pairplot(musteriler[["Alisveris_Sikligi", "Ortalama_Sepet_Tutari", "Urun_Cesidi"]], diag_kind="hist")
plt.suptitle("Müşteri Alışveriş Değişkenlerinin Dağılımı", y=1.02, fontsize=14, fontweight='bold')
plt.show()

### 4. Ölçekleme ve K-Means Kümeleme / Scaling & K-Means

In [ ]:
ozellikler = ["Alisveris_Sikligi", "Ortalama_Sepet_Tutari", "Urun_Cesidi"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(musteriler[ozellikler])

# Elbow ve Silhouette ile en iyi k değerini test edelim
wcss = []
sil_list = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_list.append(silhouette_score(X_scaled, labels))

# K=3 ile nihai kümeleme
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
musteriler['Segment'] = kmeans.fit_predict(X_scaled)

print("Segment Başına Ortalama Değerler:")
display(musteriler.groupby('Segment')[ozellikler].mean())

### 5. Alışveriş Sepetlerinin Hazırlanması / Basket Generation

In [ ]:
urunler = ['süt', 'ekmek', 'yumurta', 'peynir', 'kahve', 'çay', 'makarna', 'deterjan']

# Müşterilerin geçmiş işlem sepetlerini simüle edelim
baskets = []
for _, row in musteriler.iterrows():
    n_items = int(row['Urun_Cesidi'])
    for _ in range(max(1, int(row['Alisveris_Sikligi']))):
        # Bazı ürünlerin birlikte alınma olasılığını kural gereği artıralım
        chosen = list(np.random.choice(urunler, size=min(n_items, len(urunler)), replace=False))
        if 'kahve' in chosen and np.random.rand() > 0.3:
            if 'süt' not in chosen: chosen.append('süt')
        if 'ekmek' in chosen and np.random.rand() > 0.3:
            if 'peynir' not in chosen: chosen.append('peynir')
        baskets.append(chosen)

print(f"Toplam üretilen işlem (sepet) sayısı: {len(baskets)}")
print("İlk 3 sepet örneği:", baskets[:3])

### 6. Apriori Algoritması ve Birliktelik Kuralları / Association Rules Mining

In [ ]:
# One-Hot Matrisine dönüştürme
te = TransactionEncoder()
te_ary = te.fit(baskets).transform(baskets)
df_baskets = pd.DataFrame(te_ary, columns=te.columns_)

# Apriori ile sık öğe kümeleri (Frequent Itemsets)
frequent_itemsets = apriori(df_baskets, min_support=0.08, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.30)

# Lift değerine göre sıralama
sorted_rules = rules.sort_values(by="lift", ascending=False).reset_index(drop=True)
sorted_rules['Ürün A'] = sorted_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
sorted_rules['Ürün B'] = sorted_rules['consequents'].apply(lambda x: ', '.join(list(x)))

print(f"Bulunan toplam birliktelik kural sayısı: {len(sorted_rules)}")
display(sorted_rules[['Ürün A', 'Ürün B', 'support', 'confidence', 'lift']].head(10))

### 7. En Güçlü Ürün Birliktelikleri Görselleştirmesi / Top Rules Visualization

In [ ]:
top_rules = sorted_rules.head(15).copy()
top_rules['Kural'] = top_rules['Ürün A'] + " -> " + top_rules['Ürün B']

plt.figure(figsize=(12, 7))
sns.barplot(data=top_rules, x='Lift', y='Kural', palette='magma')
plt.title('En Güçlü Ürün Birliktelikleri (Top 15 - Lift Değerine Göre)', fontsize=15, fontweight='bold')
plt.xlabel('Lift (Kaldıraç - Birlikte Satılma Olasılığı Artışı)', fontsize=12)
plt.ylabel('Birlikte Satın Alınan Ürünler (Ürün A -> Ürün B)', fontsize=12)
plt.tight_layout()
plt.show()

### 8. Sonuç ve Stratejik Öneriler / Strategic Recommendations

- **K-Means Kümeleri:** VIP müşteri segmenti yüksek harcama ve sepet hacmiyle özel sadakat ve öncelikli hizmet stratejilerine uygun bulunmuştur.
- **Apriori Kuralları:** Yüksek Lift oranına sahip ürün ikilileri (Örn: Ekmek & Peynir, Kahve & Süt) mağaza yerleşiminde birbirine yakın konumlandırılmalı veya e-ticaret sepet önerilerinde çapraz promosyon olarak sunulmalıdır.